[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/dlt-certified/blob/main/notebooks/day-03-destinations.ipynb)

---
# Day 3 · Destinations and Loading Strategies
**certified-journeys / dlt-certified** &nbsp;|&nbsp; Data Loading

> **Goal for today:** Connect dlt to DuckDB, understand all three write dispositions (`append`, `replace`, `merge`), configure primary keys for deduplication, and use `pipeline.last_trace` to inspect what actually happened during a load.

---
## The three loading strategies

Every `@dlt.resource` has a `write_disposition` that controls how rows land in the destination. Choosing the wrong one is the #1 source of subtle data quality bugs.

| Disposition | What happens on each run | When to use it |
|---|---|---|
| `append` | New rows added to the table — existing rows untouched | Event logs, immutable facts, audit trails |
| `replace` | Table is **truncated**, then all rows inserted fresh | Small reference tables, full snapshots |
| `merge` | Rows are **upserted** using `primary_key` — new rows inserted, matching rows updated | Any slowly-changing entity with a stable ID |

### Why `merge` is the most powerful

`merge` makes your pipeline **idempotent** — you can run it 10 times with the same data and end up with exactly the same result. It's the right default for most entity tables (users, orders, products).

```
Run 1: INSERT 100 rows
Run 2: same 100 rows  →  still 100 rows (no duplicates)
Run 3: 100 rows + 5 updated  →  100 rows with 5 updated in-place
```

Under the hood dlt uses a `staging` table + `DELETE WHERE id IN (...)` + bulk `INSERT` — one atomic operation.

---
## `dataset_name` scoping

Every pipeline has a `dataset_name`. This maps to a **schema** in DuckDB (or a schema/dataset in cloud warehouses). Use it to:
- Separate environments (`raw`, `staging`, `prod`)
- Isolate pipeline runs from each other
- Scope permissions in cloud destinations

```python
# Raw ingestion layer
pipeline = dlt.pipeline(pipeline_name="crm", destination="duckdb", dataset_name="raw_crm")

# Transformed/production layer
pipeline = dlt.pipeline(pipeline_name="crm_prod", destination="duckdb", dataset_name="crm")
```

---
## Step 1 · Install dlt

In [ ]:
%pip install -q "dlt[duckdb]"

---
## Step 2 · Demonstrate `append` — rows accumulate on every run

In [ ]:
import dlt

# Helper: print row count for a table
def row_count(pipeline, full_table_name):
    with pipeline.sql_client() as client:
        with client.execute_query(f"SELECT COUNT(*) AS n FROM {full_table_name}") as cur:
            return cur.df()["n"][0]


@dlt.resource(write_disposition="append")
def events():
    """Simulates an event stream — new rows on every run."""
    yield [
        {"event_id": 1, "type": "click",    "user_id": 42, "ts": "2024-01-01T10:00:00"},
        {"event_id": 2, "type": "purchase", "user_id": 7,  "ts": "2024-01-01T10:05:00"},
        {"event_id": 3, "type": "click",    "user_id": 42, "ts": "2024-01-01T10:06:00"},
    ]


p_append = dlt.pipeline(
    pipeline_name="demo_append",
    destination="duckdb",
    dataset_name="demo",
)

# First run
p_append.run(events())
count1 = row_count(p_append, "demo.events")
print(f"After run 1: {count1} rows")

# Second run — same data
p_append.run(events())
count2 = row_count(p_append, "demo.events")
print(f"After run 2: {count2} rows  ← doubled! append adds every time")

**What just happened?**
- `append` never removes rows. 3 rows on run 1, 6 rows after run 2.
- This is correct for event logs where you *want* every occurrence.
- For entities (users, orders) this would create duplicates — use `replace` or `merge` instead.

---
## Step 3 · Demonstrate `replace` — table is wiped on every run

In [ ]:
@dlt.resource(write_disposition="replace")
def products():
    """Reference table — safe to reload from scratch on every run."""
    yield [
        {"id": 1, "name": "Widget A", "price": 9.99},
        {"id": 2, "name": "Widget B", "price": 14.99},
        {"id": 3, "name": "Gadget X", "price": 49.99},
    ]


p_replace = dlt.pipeline(
    pipeline_name="demo_replace",
    destination="duckdb",
    dataset_name="demo_r",
)

# First run
p_replace.run(products())
count1 = row_count(p_replace, "demo_r.products")
print(f"After run 1: {count1} rows")

# Second run — same data
p_replace.run(products())
count2 = row_count(p_replace, "demo_r.products")
print(f"After run 2: {count2} rows  ← same! replace truncates first")

**What just happened?**
- `replace` deleted all existing rows, then inserted fresh. Row count stays at 3 no matter how many times you run.
- This is safe when the source is the single source of truth and you always re-fetch everything.
- **Caution:** `replace` is an all-or-nothing swap. If the load fails halfway, the table is empty. For large tables, consider `merge` instead.

---
## Step 4 · Demonstrate `merge` — the idempotent upsert

In [ ]:
# First batch of users — this is what the API returns on run 1
USERS_V1 = [
    {"id": 1, "name": "Alice",   "status": "active",   "score": 80},
    {"id": 2, "name": "Bob",     "status": "active",   "score": 65},
    {"id": 3, "name": "Charlie", "status": "inactive", "score": 90},
]

# Second batch — same users, but Alice's score changed and Dave is new
USERS_V2 = [
    {"id": 1, "name": "Alice",   "status": "active",   "score": 95},  # updated score
    {"id": 2, "name": "Bob",     "status": "active",   "score": 65},  # unchanged
    {"id": 3, "name": "Charlie", "status": "inactive", "score": 90},  # unchanged
    {"id": 4, "name": "Dave",    "status": "active",   "score": 72},  # new user
]


def make_users_resource(data):
    @dlt.resource(
        name="users",
        primary_key="id",              # ← merge key: rows with same id are upserted
        write_disposition="merge",
    )
    def users():
        yield data
    return users


p_merge = dlt.pipeline(
    pipeline_name="demo_merge",
    destination="duckdb",
    dataset_name="demo_m",
)

# Run 1: load the original 3 users
p_merge.run(make_users_resource(USERS_V1)())
count1 = row_count(p_merge, "demo_m.users")
print(f"After run 1: {count1} rows")

# Show Alice's score after run 1
with p_merge.sql_client() as client:
    with client.execute_query("SELECT id, name, score FROM demo_m.users ORDER BY id") as cur:
        print("Run 1 state:")
        print(cur.df().to_string(index=False))

In [ ]:
# Run 2: load the updated batch
p_merge.run(make_users_resource(USERS_V2)())
count2 = row_count(p_merge, "demo_m.users")
print(f"After run 2: {count2} rows  ← 4 (Dave added, no duplicates)")

with p_merge.sql_client() as client:
    with client.execute_query("SELECT id, name, score, status FROM demo_m.users ORDER BY id") as cur:
        print("Run 2 state (Alice's score updated to 95, Dave added):")
        print(cur.df().to_string(index=False))

**What just happened?**
- Run 1: 3 rows inserted.
- Run 2: Alice (id=1) was **updated** (score 80 → 95), Dave (id=4) was **inserted**. Total: 4 rows.
- Bob and Charlie were re-inserted but their values didn't change — the net effect is identical.
- Running run 2 a second time would still give 4 rows: **idempotent**.

---
## Step 5 · Composite primary keys

Some entities don't have a single unique ID — they're unique on a **combination** of fields. dlt handles this with a list primary key.

In [ ]:
# Sales data: unique per (region, product_id, date)
# There is no single 'id' column — the combination is the key.

@dlt.resource(
    name="daily_sales",
    primary_key=["region", "product_id", "date"],  # composite PK
    write_disposition="merge",
)
def daily_sales_v1():
    yield [
        {"region": "us", "product_id": 1, "date": "2024-01-01", "units": 50,  "revenue": 499.50},
        {"region": "us", "product_id": 2, "date": "2024-01-01", "units": 30,  "revenue": 449.70},
        {"region": "eu", "product_id": 1, "date": "2024-01-01", "units": 20,  "revenue": 199.80},
    ]


@dlt.resource(
    name="daily_sales",
    primary_key=["region", "product_id", "date"],
    write_disposition="merge",
)
def daily_sales_v2():
    # Corrected revenue for us/product_id=1, new record for eu/product_id=2
    yield [
        {"region": "us", "product_id": 1, "date": "2024-01-01", "units": 50,  "revenue": 512.00},  # corrected
        {"region": "eu", "product_id": 2, "date": "2024-01-01", "units": 10,  "revenue": 149.90},  # new
    ]


p_composite = dlt.pipeline(
    pipeline_name="composite_pk",
    destination="duckdb",
    dataset_name="sales",
)

p_composite.run(daily_sales_v1())
print(f"After run 1: {row_count(p_composite, 'sales.daily_sales')} rows")

p_composite.run(daily_sales_v2())
print(f"After run 2: {row_count(p_composite, 'sales.daily_sales')} rows")

with p_composite.sql_client() as client:
    with client.execute_query(
        "SELECT region, product_id, date, units, revenue FROM sales.daily_sales ORDER BY region, product_id"
    ) as cur:
        print("\nFinal state (us/p1 revenue corrected, eu/p2 inserted):")
        print(cur.df().to_string(index=False))

**What just happened?**
- Run 1 loaded 3 rows.
- Run 2 sent 2 rows: one update (us/1/2024-01-01 revenue corrected) and one new row (eu/2/2024-01-01).
- Final state: 4 rows — no duplicates, corrected values in place.
- The composite `primary_key=["region", "product_id", "date"]` is what makes this work.

---
## Step 6 · Inspect load traces with `pipeline.last_trace`

Every run produces a **trace** — a structured log of what happened. Use it to debug, audit, and monitor your pipelines.

In [ ]:
# Run a pipeline to generate a fresh trace
@dlt.resource(primary_key="id", write_disposition="merge")
def trace_demo():
    yield [
        {"id": 1, "val": "alpha"},
        {"id": 2, "val": "beta"},
        {"id": 3, "val": "gamma"},
    ]


p_trace = dlt.pipeline(
    pipeline_name="trace_demo",
    destination="duckdb",
    dataset_name="td",
)

p_trace.run(trace_demo())

# Access the last trace
trace = p_trace.last_trace
print("=== Load Trace ===")
print(f"Pipeline name  : {trace.pipeline_name}")
print(f"Started at     : {trace.started_at}")
print(f"Finished at    : {trace.finished_at}")
print(f"Status         : {'SUCCESS' if trace.last_action_succeeded else 'FAILED'}")

In [ ]:
# Drill into the load steps
load_info = trace.last_normalize_info
if load_info:
    print("Normalize step info:")
    print(load_info)

# The load step contains per-table metrics
for step in trace.steps:
    print(f"\nStep: {step.step}  |  Status: {step.step_exception or 'OK'}")
    if hasattr(step, 'step_info') and step.step_info:
        print(f"  Info: {step.step_info}")

In [ ]:
# The _dlt_loads table in DuckDB is the persistent load history — 
# every run writes a record here regardless of trace API.

with p_trace.sql_client() as client:
    with client.execute_query(
        "SELECT load_id, status, inserted_at FROM td._dlt_loads ORDER BY inserted_at DESC LIMIT 5"
    ) as cur:
        print("Load history from _dlt_loads table:")
        print(cur.df().to_string(index=False))

**What just happened?**
- `pipeline.last_trace` gives you the in-memory trace object from the most recent run.
- The `_dlt_loads` system table is the **persistent** load history — it survives process restarts and is stored in the destination itself.
- `status=0` means success; any non-zero value is a failure code.

---
## Step 7 · Write disposition comparison side by side

In [ ]:
import dlt

SAMPLE_DATA = [
    {"id": 1, "value": "original"},
    {"id": 2, "value": "original"},
]

UPDATED_DATA = [
    {"id": 1, "value": "updated"},
    {"id": 3, "value": "new"},
]

summary = []

for disposition in ["append", "replace", "merge"]:
    @dlt.resource(name="items", primary_key="id", write_disposition=disposition)
    def items_resource(data=SAMPLE_DATA):
        yield data

    p = dlt.pipeline(
        pipeline_name=f"compare_{disposition}",
        destination="duckdb",
        dataset_name="cmp",
    )

    p.run(items_resource(SAMPLE_DATA))  # run 1
    p.run(items_resource(UPDATED_DATA)) # run 2

    with p.sql_client() as client:
        with client.execute_query("SELECT id, value FROM cmp.items ORDER BY id") as cur:
            rows = cur.df()
            summary.append({
                "disposition": disposition,
                "row_count": len(rows),
                "id_1_value": rows[rows["id"]==1]["value"].values[0] if len(rows[rows["id"]==1]) > 0 else "—",
            })
        
print("\n=== Write Disposition Comparison ===")
print(f"{'Disposition':<12} {'Row count':>10} {'id=1 value after run 2':>25}")
print("-" * 52)
for s in summary:
    print(f"{s['disposition']:<12} {s['row_count']:>10} {s['id_1_value']:>25}")

Reading the comparison table:
- **append**: 4 rows (2 from run 1 + 2 from run 2). id=1 appears twice with both values.
- **replace**: 2 rows (run 2 wiped run 1). id=1 shows `updated`, id=2 is gone.
- **merge**: 3 rows (id=1 updated in-place, id=2 unchanged, id=3 inserted). id=1 shows `updated`.

---
## Challenge — do this yourself

You have order data that arrives in daily batches. Build a pipeline that:

1. Creates an `orders` resource with `write_disposition="merge"` and `primary_key="order_id"`.
2. Loads the first batch (3 orders).
3. Loads a second batch where one order's `status` has changed from `"pending"` to `"shipped"` and one new order is added.
4. Prints the final row count (should be 4) and the updated status.
5. Bonus: query `_dlt_loads` and print how many load runs have been recorded.

In [ ]:
# Your solution here


<details>
<summary>Show solution</summary>

```python
import dlt

BATCH_1 = [
    {"order_id": "ORD-001", "customer": "Alice", "amount": 99.99,  "status": "pending"},
    {"order_id": "ORD-002", "customer": "Bob",   "amount": 199.00, "status": "shipped"},
    {"order_id": "ORD-003", "customer": "Carol", "amount": 49.50,  "status": "pending"},
]

BATCH_2 = [
    {"order_id": "ORD-001", "customer": "Alice", "amount": 99.99,  "status": "shipped"},  # updated
    {"order_id": "ORD-004", "customer": "Dave",  "amount": 310.00, "status": "pending"},  # new
]

def make_orders(data):
    @dlt.resource(name="orders", primary_key="order_id", write_disposition="merge")
    def orders():
        yield data
    return orders

p = dlt.pipeline(pipeline_name="challenge_orders", destination="duckdb", dataset_name="ch")
p.run(make_orders(BATCH_1)())
p.run(make_orders(BATCH_2)())

with p.sql_client() as client:
    with client.execute_query("SELECT COUNT(*) AS n FROM ch.orders") as cur:
        print(f"Row count: {cur.df()['n'][0]}")  # expect 4
    with client.execute_query("SELECT order_id, status FROM ch.orders ORDER BY order_id") as cur:
        print(cur.df().to_string(index=False))
    # Bonus
    with client.execute_query("SELECT COUNT(*) AS loads FROM ch._dlt_loads") as cur:
        print(f"Load runs recorded: {cur.df()['loads'][0]}")
```
</details>

---
## Day 3 key concepts recap

| Concept | What to remember |
|---|---|
| `append` | Rows accumulate — correct for events and logs, wrong for entities |
| `replace` | Atomic full reload — simple, but all-or-nothing |
| `merge` | Idempotent upsert — the right default for most entity tables |
| `primary_key="id"` | Single-column merge key — matches rows for update-or-insert |
| `primary_key=["a","b"]` | Composite merge key — when uniqueness spans multiple columns |
| `dataset_name` | Logical namespace — maps to a DuckDB schema or cloud warehouse dataset |
| `pipeline.last_trace` | In-memory audit log of the last run's steps and timing |
| `_dlt_loads` table | Persistent load history stored in the destination itself |

> **Tip:** `merge` with composite primary keys is the most powerful dlt feature for idempotent loads. Make it your default until you have a reason not to.

---
## What's next → Day 4

**Day 4** → Incremental loading and state management: cursor fields, `dlt.sources.incremental`, and how dlt persists state in the destination so you never need an external database.

Mark Day 3 complete in your [tracker](../index.html).